# Notebook 2 – Cleaning, NER, and relation extraction

In this notebook, we transform the raw movie summaries collected in Notebook 1 into structured information that can be used to build the initial knowledge graph.

This notebook has four goals:
1. Load the crawled movie metadata and Wikipedia summaries.
2. Clean and filter the textual summaries.
3. Apply Named Entity Recognition (NER) with spaCy.
4. Extract simple candidate relations using dependency parsing.

The outputs of this notebook will be saved for the next stages:
- `data/interim/cleaned_plots.jsonl`
- `data/interim/extracted_entities.csv`
- `data/interim/extracted_relations.csv`
- `data/interim/extracted_knowledge.csv`

> Note: This notebook is designed to be lightweight and Colab-friendly. It uses `en_core_web_sm` by default for speed. If you want stronger NLP quality and have time/resources, you can switch to `en_core_web_trf`.

In [1]:
# Cell 2 — Imports, paths, and load raw data from Notebook 1

# Run once in Colab if needed, then leave commented
# %pip install -q spacy pandas

import os
import re
import json
import pandas as pd
import spacy
from spacy.cli import download as spacy_download

# Create output folder for Notebook 2 artifacts
os.makedirs("data/interim", exist_ok=True)

# Input files produced by Notebook 1
films_path = "/content/wikidata_films.json"
plots_path = "/content/wiki_plots.jsonl"

# Check that Notebook 1 outputs exist
if not os.path.exists(films_path):
    raise FileNotFoundError(
        f"Missing file: {films_path}\nRun Notebook 1 first to generate wikidata_films.json."
    )

if not os.path.exists(plots_path):
    raise FileNotFoundError(
        f"Missing file: {plots_path}\nRun Notebook 1 first to generate wiki_plots.jsonl."
    )

# Load structured film metadata
with open(films_path, "r", encoding="utf-8") as f:
    films = json.load(f)

# Load JSONL plot summaries
plots = []
with open(plots_path, "r", encoding="utf-8") as f:
    for line_number, line in enumerate(f, start=1):
        line = line.strip()
        if not line:
            continue
        try:
            plots.append(json.loads(line))
        except json.JSONDecodeError as e:
            print(f"Skipping malformed JSON on line {line_number}: {e}")

# Convert to DataFrames
films_df = pd.DataFrame(films)
plots_df = pd.DataFrame(plots)

print("=== Raw data loaded ===")
print(f"Films loaded: {len(films_df)}")
print(f"Plot summaries loaded: {len(plots_df)}")
print("\nFilm columns:")
print(list(films_df.columns))
print("\nPlot columns:")
print(list(plots_df.columns))

print("\nFilms preview:")
display(films_df.head(2))

print("\nPlots preview:")
display(plots_df.head(2))

=== Raw data loaded ===
Films loaded: 998
Plot summaries loaded: 649

Film columns:
['uri', 'id', 'label', 'wikipedia_title', 'date', 'directors', 'genres', 'countries', 'imdb_ids', 'cast', 'awards', 'production_companies', 'followed_by', 'preceded_by']

Plot columns:
['id', 'title', 'wikidata_uri', 'summary', 'page_url']

Films preview:


,uri,id,label,wikipedia_title,date,directors,genres,countries,imdb_ids,cast,awards,production_companies,followed_by,preceded_by
0,http://www.wikidata.org/entity/Q138793787,Q138793787,"1+1+1 Life, Love, Chaos","1+1+1 Life, Love, Chaos",None,[],[],[],[],[],[],[],[],[]
1,http://www.wikidata.org/entity/Q134083298,Q134083298,1-800-On-Her-Own,1-800-On-Her-Own,None,[Dana Flor],[],[United States],[tt32147765],[],[],[],[],[]



Plots preview:


,id,title,wikidata_uri,summary,page_url
0,Q130295240,Ajab Raat Ni Gajab Vaat,http://www.wikidata.org/entity/Q130295240,Ajab Raat Ni Gajab Vaat is a 2024 Gujarati com...,https://en.wikipedia.org/wiki/Ajab_Raat_Ni_Gaj...
1,Q130245494,Alanaati Ramchandrudu,http://www.wikidata.org/entity/Q130245494,Alanaati Ramchandrudu is a 2024 Indian Telugu-...,https://en.wikipedia.org/wiki/Alanaati_Ramchan...


In [2]:
# Cell 3 — Define text cleaning and usefulness filters

MIN_WORDS = 25

def clean_text(text: str) -> str:
    """
    Normalize summary text so it is easier to process with spaCy.
    """
    if not isinstance(text, str):
        return ""

    text = text.replace("\xa0", " ")                 # non-breaking spaces
    text = re.sub(r"\[[^\]]*\]", " ", text)          # remove bracketed refs like [1]
    text = re.sub(r"\([^)]*listen[^)]*\)", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"\s+", " ", text).strip()         # collapse repeated whitespace
    return text

def word_count(text: str) -> int:
    """
    Count whitespace-separated words in a string.
    """
    if not isinstance(text, str):
        return 0
    return len(text.split())

def is_useful_text(text: str, min_words: int = MIN_WORDS) -> bool:
    """
    Keep only non-empty summaries with enough words to be useful for IE.
    """
    cleaned = clean_text(text)
    return word_count(cleaned) >= min_words

In [3]:
# Cell 4 — Apply cleaning pipeline

cleaned_records = []

for row in plots_df.to_dict(orient="records"):
    cleaned_summary = clean_text(row.get("summary", ""))

    if is_useful_text(cleaned_summary, min_words=MIN_WORDS):
        cleaned_records.append({
            "id": row.get("id"),
            "title": row.get("title"),
            "wikidata_uri": row.get("wikidata_uri"),
            "summary": cleaned_summary,
            "page_url": row.get("page_url", ""),
        })

# Build cleaned dataframe
cleaned_plots_df = pd.DataFrame(cleaned_records)

before_dedup = len(cleaned_plots_df)

# Drop duplicates by Wikidata URI if available
if "wikidata_uri" in cleaned_plots_df.columns and not cleaned_plots_df.empty:
    cleaned_plots_df = (
        cleaned_plots_df
        .drop_duplicates(subset=["wikidata_uri"])
        .reset_index(drop=True)
    )

after_dedup = len(cleaned_plots_df)

# Save cleaned summaries
cleaned_path = "/content/cleaned_plots.jsonl"
with open(cleaned_path, "w", encoding="utf-8") as f:
    for rec in cleaned_plots_df.to_dict(orient="records"):
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

# Stats
raw_count = len(plots_df)
kept_before_dedup = before_dedup
kept_after_dedup = after_dedup
dropped_by_filter = raw_count - kept_before_dedup
dropped_by_dedup = kept_before_dedup - kept_after_dedup
avg_word_count = (
    cleaned_plots_df["summary"].apply(word_count).mean()
    if not cleaned_plots_df.empty else 0
)

print("=== Cleaning summary ===")
print(f"Raw summaries: {raw_count}")
print(f"Kept after filtering: {kept_before_dedup}")
print(f"Kept after deduplication: {kept_after_dedup}")
print(f"Dropped by filter: {dropped_by_filter}")
print(f"Dropped by deduplication: {dropped_by_dedup}")
print(f"Average word count: {avg_word_count:.1f}")
print(f"Saved cleaned summaries to: {cleaned_path}")

display(cleaned_plots_df.head(3))

=== Cleaning summary ===
Raw summaries: 649
Kept after filtering: 580
Kept after deduplication: 580
Dropped by filter: 69
Dropped by deduplication: 0
Average word count: 56.5
Saved cleaned summaries to: /content/cleaned_plots.jsonl


,id,title,wikidata_uri,summary,page_url
0,Q130295240,Ajab Raat Ni Gajab Vaat,http://www.wikidata.org/entity/Q130295240,Ajab Raat Ni Gajab Vaat is a 2024 Gujarati com...,https://en.wikipedia.org/wiki/Ajab_Raat_Ni_Gaj...
1,Q130245494,Alanaati Ramchandrudu,http://www.wikidata.org/entity/Q130245494,Alanaati Ramchandrudu is a 2024 Indian Telugu-...,https://en.wikipedia.org/wiki/Alanaati_Ramchan...
2,Q135901978,Angammal,http://www.wikidata.org/entity/Q135901978,Angammal is a 2025 Indian Tamil-language drama...,https://en.wikipedia.org/wiki/Angammal


In [4]:
# Cell 5 — Load spaCy model

MODEL_NAME = "en_core_web_sm"

try:
    nlp = spacy.load(MODEL_NAME)
except OSError:
    print(f"spaCy model '{MODEL_NAME}' not found. Downloading...")
    spacy_download(MODEL_NAME)
    nlp = spacy.load(MODEL_NAME)

print(f"Loaded spaCy model: {MODEL_NAME}")
print("spaCy pipeline:", nlp.pipe_names)

Loaded spaCy model: en_core_web_sm
spaCy pipeline: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


In [5]:
# Cell 6 — Define NER extraction logic

TARGET_ENTITY_LABELS = {"PERSON", "ORG", "GPE", "DATE"}

def extract_entities_from_doc(doc, title: str, film_id: str, wikidata_uri: str):
    rows = []

    for sent in doc.sents:
        sentence_text = sent.text.strip()

        for ent in sent.ents:
            entity_text = ent.text.strip()

            if ent.label_ in TARGET_ENTITY_LABELS and entity_text:
                rows.append({
                    "film_id": film_id,
                    "title": title,
                    "wikidata_uri": wikidata_uri,
                    "entity_text": entity_text,
                    "entity_label": ent.label_,
                    "sentence": sentence_text,
                    "start_char": ent.start_char,
                    "end_char": ent.end_char,
                })

    return rows

In [6]:
# Cell 7 — Run NER over cleaned summaries

from tqdm.auto import tqdm

entity_rows = []

for row in tqdm(cleaned_plots_df.to_dict(orient="records"), desc="Running NER"):
    doc = nlp(row["summary"])
    entity_rows.extend(
        extract_entities_from_doc(
            doc=doc,
            title=row["title"],
            film_id=row["id"],
            wikidata_uri=row["wikidata_uri"],
        )
    )

entities_df = pd.DataFrame(entity_rows)

if not entities_df.empty:
    entities_df = entities_df.drop_duplicates().reset_index(drop=True)

print(f"Extracted {len(entities_df)} entity mentions.")

if not entities_df.empty:
    print("\nEntity label counts:")
    display(entities_df["entity_label"].value_counts().rename_axis("label").reset_index(name="count"))
    display(entities_df.head(10))
else:
    print("No entities were extracted.")

Running NER:   0%|          | 0/580 [00:00<?, ?it/s]

Extracted 5262 entity mentions.

Entity label counts:


,label,count
0,PERSON,3122
1,ORG,945
2,DATE,770
3,GPE,425


,film_id,title,wikidata_uri,entity_text,entity_label,sentence,start_char,end_char
0,Q130295240,Ajab Raat Ni Gajab Vaat,http://www.wikidata.org/entity/Q130295240,Raat Ni Gajab Vaat,PERSON,Ajab Raat Ni Gajab Vaat is a 2024 Gujarati com...,5,23
1,Q130295240,Ajab Raat Ni Gajab Vaat,http://www.wikidata.org/entity/Q130295240,2024,DATE,Ajab Raat Ni Gajab Vaat is a 2024 Gujarati com...,29,33
2,Q130295240,Ajab Raat Ni Gajab Vaat,http://www.wikidata.org/entity/Q130295240,Prem Gadhavi & Killol Parmar,ORG,Ajab Raat Ni Gajab Vaat is a 2024 Gujarati com...,69,97
3,Q130295240,Ajab Raat Ni Gajab Vaat,http://www.wikidata.org/entity/Q130295240,Prem Gadhavi,PERSON,Ajab Raat Ni Gajab Vaat is a 2024 Gujarati com...,113,125
4,Q130295240,Ajab Raat Ni Gajab Vaat,http://www.wikidata.org/entity/Q130295240,Aditi Varma & Nikita Shah,ORG,Ajab Raat Ni Gajab Vaat is a 2024 Gujarati com...,127,152
5,Q130295240,Ajab Raat Ni Gajab Vaat,http://www.wikidata.org/entity/Q130295240,Bhavya Gandhi,PERSON,"It stars Bhavya Gandhi Aarohi Patel, Deep Vaid...",163,176
6,Q130295240,Ajab Raat Ni Gajab Vaat,http://www.wikidata.org/entity/Q130295240,Patel,PERSON,"It stars Bhavya Gandhi Aarohi Patel, Deep Vaid...",184,189
7,Q130295240,Ajab Raat Ni Gajab Vaat,http://www.wikidata.org/entity/Q130295240,Deep Vaidya,PERSON,"It stars Bhavya Gandhi Aarohi Patel, Deep Vaid...",191,202
8,Q130295240,Ajab Raat Ni Gajab Vaat,http://www.wikidata.org/entity/Q130295240,Radhika Barot & RJ Harsh,PERSON,"It stars Bhavya Gandhi Aarohi Patel, Deep Vaid...",204,228
9,Q130295240,Ajab Raat Ni Gajab Vaat,http://www.wikidata.org/entity/Q130295240,Jayesh Pavra,PERSON,The film is produced by Dr. Jayesh Pavra,258,270


In [7]:
# Cell 8 — Define simple dependency-based relation extraction

def find_covering_entity(token, sent_ents):
    """
    Return the entity text whose span covers the token, or None.
    """
    for ent in sent_ents:
        if ent.start <= token.i < ent.end:
            return ent.text.strip()
    return None


def extract_relations_from_doc(doc, title: str, film_id: str, wikidata_uri: str):
    rows = []
    seen = set()

    for sent in doc.sents:
        sentence_text = sent.text.strip()
        sent_ents = [ent for ent in sent.ents if ent.label_ in TARGET_ENTITY_LABELS]

        if len(sent_ents) < 2:
            continue

        found_relation = False

        for token in sent:
            if token.pos_ != "VERB":
                continue

            predicate = token.lemma_.strip().lower()
            if not predicate:
                continue

            subj_entity = None
            obj_entity = None

            for child in token.children:
                if child.dep_ in ("nsubj", "nsubjpass"):
                    subj_entity = find_covering_entity(child, sent_ents)
                elif child.dep_ in ("dobj", "obj", "pobj", "attr"):
                    obj_entity = find_covering_entity(child, sent_ents)

            if subj_entity and obj_entity and subj_entity != obj_entity:
                key = (film_id, subj_entity, predicate, obj_entity, sentence_text)
                if key not in seen:
                    rows.append({
                        "film_id": film_id,
                        "title": title,
                        "wikidata_uri": wikidata_uri,
                        "subject": subj_entity,
                        "predicate": predicate,
                        "object": obj_entity,
                        "sentence": sentence_text,
                        "extraction_method": "dependency",
                    })
                    seen.add(key)
                found_relation = True

        # Fallback: use root verb + first two entities if nothing explicit was found
        if not found_relation:
            root_verbs = [tok for tok in sent if tok.dep_ == "ROOT" and tok.pos_ == "VERB"]
            if root_verbs and len(sent_ents) >= 2:
                predicate = root_verbs[0].lemma_.strip().lower()
                subj_entity = sent_ents[0].text.strip()
                obj_entity = sent_ents[1].text.strip()

                if predicate and subj_entity and obj_entity and subj_entity != obj_entity:
                    key = (film_id, subj_entity, predicate, obj_entity, sentence_text)
                    if key not in seen:
                        rows.append({
                            "film_id": film_id,
                            "title": title,
                            "wikidata_uri": wikidata_uri,
                            "subject": subj_entity,
                            "predicate": predicate,
                            "object": obj_entity,
                            "sentence": sentence_text,
                            "extraction_method": "root_verb_fallback",
                        })
                        seen.add(key)

    return rows

In [8]:
# Cell 9 — Run relation extraction

from tqdm.auto import tqdm

relation_rows = []

for row in tqdm(cleaned_plots_df.to_dict(orient="records"), desc="Extracting relations"):
    doc = nlp(row["summary"])
    relation_rows.extend(
        extract_relations_from_doc(
            doc=doc,
            title=row["title"],
            film_id=row["id"],
            wikidata_uri=row["wikidata_uri"],
        )
    )

relations_df = pd.DataFrame(relation_rows)

if not relations_df.empty:
    relations_df = relations_df.drop_duplicates().reset_index(drop=True)

print(f"Extracted {len(relations_df)} candidate relations.")

if not relations_df.empty:
    display(relations_df.head(10))
else:
    print("No relations were extracted.")

Extracting relations:   0%|          | 0/580 [00:00<?, ?it/s]

Extracted 655 candidate relations.


,film_id,title,wikidata_uri,subject,predicate,object,sentence,extraction_method
0,Q130295240,Ajab Raat Ni Gajab Vaat,http://www.wikidata.org/entity/Q130295240,Bhavya Gandhi,star,Patel,"It stars Bhavya Gandhi Aarohi Patel, Deep Vaid...",root_verb_fallback
1,Q130245494,Alanaati Ramchandrudu,http://www.wikidata.org/entity/Q130245494,Hymavathi Jadapolu,produce,Hyniva Creations LLP,"It is produced by Hymavathi Jadapolu, Sreeram ...",root_verb_fallback
2,Q130245494,Alanaati Ramchandrudu,http://www.wikidata.org/entity/Q130245494,Krishna Vamsi,feature,Mokksha,"the film features Krishna Vamsi, Mokksha, Brah...",root_verb_fallback
3,Q123185887,Anora,http://www.wikidata.org/entity/Q123185887,Mikey Madison,star,"Anora ""Ani"" Mikheeva","It stars Mikey Madison as Anora ""Ani"" Mikheeva...",root_verb_fallback
4,Q123185887,Anora,http://www.wikidata.org/entity/Q123185887,Yura Borisov,include,Karren Karagulian,"The supporting cast includes Yura Borisov, Kar...",root_verb_fallback
5,Q124532587,Anweshippin Kandethum,http://www.wikidata.org/entity/Q124532587,Darwin,produce,Dolwin Kuriakose,The film is produced by Darwin and Dolwin Kuri...,root_verb_fallback
6,Q124532587,Anweshippin Kandethum,http://www.wikidata.org/entity/Q124532587,Tovino Thomas,feature,Siddique,The film features Tovino Thomas in the lead ro...,root_verb_fallback
7,Q133895441,Any Day Now (2024 film),http://www.wikidata.org/entity/Q133895441,1990,base,Isabella Stewart Gardner Museum,It is based on the 1990 Isabella Stewart Gardn...,root_verb_fallback
8,Q130282422,Anywhere Anytime,http://www.wikidata.org/entity/Q130282422,September 2024,premier,Italy,It premiered at the 81st Venice International ...,root_verb_fallback
9,Q112865585,Apartment 7A,http://www.wikidata.org/entity/Q112865585,Natalie Erika James,star,Julia Garner,Directed and co-written by Natalie Erika James...,root_verb_fallback


In [9]:
# Cell 10 — Save cleaned text, entities, relations, and a combined knowledge table

entities_path = "data/interim/extracted_entities.csv"
relations_path = "data/interim/extracted_relations.csv"
combined_path = "data/interim/extracted_knowledge.csv"

entities_df.to_csv(entities_path, index=False)
relations_df.to_csv(relations_path, index=False)

combined_df = pd.concat([
    entities_df.assign(record_type="entity"),
    relations_df.assign(record_type="relation")
], ignore_index=True, sort=False)

combined_df.to_csv(combined_path, index=False)

print(f"Saved entities to: {entities_path}")
print(f"Saved relations to: {relations_path}")
print(f"Saved combined knowledge table to: {combined_path}")

Saved entities to: data/interim/extracted_entities.csv
Saved relations to: data/interim/extracted_relations.csv
Saved combined knowledge table to: data/interim/extracted_knowledge.csv


In [10]:
# Cell 11 — Quality checks, examples, and ambiguity candidates

print("=== Notebook 2 summary ===")
print(f"Cleaned summaries: {len(cleaned_plots_df)}")
print(f"Entity mentions: {len(entities_df)}")
print(f"Candidate relations: {len(relations_df)}")

print("\n=== Sample entities ===")
display(entities_df.head(10))

print("\n=== Sample relations ===")
display(relations_df.head(10))

# Candidate ambiguity case 1: same text assigned multiple labels
multi_label_entities = (
    entities_df.groupby("entity_text")["entity_label"]
    .nunique()
    .reset_index(name="num_labels")
)
multi_label_entities = multi_label_entities[multi_label_entities["num_labels"] > 1]

print("\n=== Candidate ambiguity cases: same entity text with multiple labels ===")
display(multi_label_entities.head(10))

# Candidate ambiguity case 2: very frequent entities across many films
frequent_entities = (
    entities_df.groupby("entity_text")["title"]
    .nunique()
    .sort_values(ascending=False)
    .reset_index(name="num_films")
)

print("\n=== Frequent entities across multiple films ===")
display(frequent_entities.head(10))

# Candidate ambiguity case 3: short or noisy entities
short_entities = entities_df[entities_df["entity_text"].str.len() <= 4]

print("\n=== Short entity candidates to inspect manually ===")
display(short_entities.head(10))

=== Notebook 2 summary ===
Cleaned summaries: 580
Entity mentions: 5262
Candidate relations: 655

=== Sample entities ===


,film_id,title,wikidata_uri,entity_text,entity_label,sentence,start_char,end_char
0,Q130295240,Ajab Raat Ni Gajab Vaat,http://www.wikidata.org/entity/Q130295240,Raat Ni Gajab Vaat,PERSON,Ajab Raat Ni Gajab Vaat is a 2024 Gujarati com...,5,23
1,Q130295240,Ajab Raat Ni Gajab Vaat,http://www.wikidata.org/entity/Q130295240,2024,DATE,Ajab Raat Ni Gajab Vaat is a 2024 Gujarati com...,29,33
2,Q130295240,Ajab Raat Ni Gajab Vaat,http://www.wikidata.org/entity/Q130295240,Prem Gadhavi & Killol Parmar,ORG,Ajab Raat Ni Gajab Vaat is a 2024 Gujarati com...,69,97
3,Q130295240,Ajab Raat Ni Gajab Vaat,http://www.wikidata.org/entity/Q130295240,Prem Gadhavi,PERSON,Ajab Raat Ni Gajab Vaat is a 2024 Gujarati com...,113,125
4,Q130295240,Ajab Raat Ni Gajab Vaat,http://www.wikidata.org/entity/Q130295240,Aditi Varma & Nikita Shah,ORG,Ajab Raat Ni Gajab Vaat is a 2024 Gujarati com...,127,152
5,Q130295240,Ajab Raat Ni Gajab Vaat,http://www.wikidata.org/entity/Q130295240,Bhavya Gandhi,PERSON,"It stars Bhavya Gandhi Aarohi Patel, Deep Vaid...",163,176
6,Q130295240,Ajab Raat Ni Gajab Vaat,http://www.wikidata.org/entity/Q130295240,Patel,PERSON,"It stars Bhavya Gandhi Aarohi Patel, Deep Vaid...",184,189
7,Q130295240,Ajab Raat Ni Gajab Vaat,http://www.wikidata.org/entity/Q130295240,Deep Vaidya,PERSON,"It stars Bhavya Gandhi Aarohi Patel, Deep Vaid...",191,202
8,Q130295240,Ajab Raat Ni Gajab Vaat,http://www.wikidata.org/entity/Q130295240,Radhika Barot & RJ Harsh,PERSON,"It stars Bhavya Gandhi Aarohi Patel, Deep Vaid...",204,228
9,Q130295240,Ajab Raat Ni Gajab Vaat,http://www.wikidata.org/entity/Q130295240,Jayesh Pavra,PERSON,The film is produced by Dr. Jayesh Pavra,258,270



=== Sample relations ===


,film_id,title,wikidata_uri,subject,predicate,object,sentence,extraction_method
0,Q130295240,Ajab Raat Ni Gajab Vaat,http://www.wikidata.org/entity/Q130295240,Bhavya Gandhi,star,Patel,"It stars Bhavya Gandhi Aarohi Patel, Deep Vaid...",root_verb_fallback
1,Q130245494,Alanaati Ramchandrudu,http://www.wikidata.org/entity/Q130245494,Hymavathi Jadapolu,produce,Hyniva Creations LLP,"It is produced by Hymavathi Jadapolu, Sreeram ...",root_verb_fallback
2,Q130245494,Alanaati Ramchandrudu,http://www.wikidata.org/entity/Q130245494,Krishna Vamsi,feature,Mokksha,"the film features Krishna Vamsi, Mokksha, Brah...",root_verb_fallback
3,Q123185887,Anora,http://www.wikidata.org/entity/Q123185887,Mikey Madison,star,"Anora ""Ani"" Mikheeva","It stars Mikey Madison as Anora ""Ani"" Mikheeva...",root_verb_fallback
4,Q123185887,Anora,http://www.wikidata.org/entity/Q123185887,Yura Borisov,include,Karren Karagulian,"The supporting cast includes Yura Borisov, Kar...",root_verb_fallback
5,Q124532587,Anweshippin Kandethum,http://www.wikidata.org/entity/Q124532587,Darwin,produce,Dolwin Kuriakose,The film is produced by Darwin and Dolwin Kuri...,root_verb_fallback
6,Q124532587,Anweshippin Kandethum,http://www.wikidata.org/entity/Q124532587,Tovino Thomas,feature,Siddique,The film features Tovino Thomas in the lead ro...,root_verb_fallback
7,Q133895441,Any Day Now (2024 film),http://www.wikidata.org/entity/Q133895441,1990,base,Isabella Stewart Gardner Museum,It is based on the 1990 Isabella Stewart Gardn...,root_verb_fallback
8,Q130282422,Anywhere Anytime,http://www.wikidata.org/entity/Q130282422,September 2024,premier,Italy,It premiered at the 81st Venice International ...,root_verb_fallback
9,Q112865585,Apartment 7A,http://www.wikidata.org/entity/Q112865585,Natalie Erika James,star,Julia Garner,Directed and co-written by Natalie Erika James...,root_verb_fallback



=== Candidate ambiguity cases: same entity text with multiple labels ===


,entity_text,num_labels
541,Baasuri Films,2
566,Baghjan,2
744,Blood Star,2
960,Chhota Bheem,2
1076,Crow,2
1176,Deadpool,2
1203,Deepavali Bonus,2
1725,Henry,2
2131,Kannada,2
2388,London Recruits,2



=== Frequent entities across multiple films ===


,entity_text,num_films
0,2024,384
1,Indian Tamil-language,23
2,the United States,14
3,France,13
4,2016,10
5,2019,10
6,Netflix,10
7,2017,9
8,2022,9
9,India,9



=== Short entity candidates to inspect manually ===


,film_id,title,wikidata_uri,entity_text,entity_label,sentence,start_char,end_char
1,Q130295240,Ajab Raat Ni Gajab Vaat,http://www.wikidata.org/entity/Q130295240,2024,DATE,Ajab Raat Ni Gajab Vaat is a 2024 Gujarati com...,29,33
12,Q130245494,Alanaati Ramchandrudu,http://www.wikidata.org/entity/Q130245494,2024,DATE,Alanaati Ramchandrudu is a 2024 Indian Telugu-...,27,31
21,Q135901978,Angammal,http://www.wikidata.org/entity/Q135901978,2025,DATE,Angammal is a 2025 Indian Tamil-language drama...,14,18
28,Q123185887,Anora,http://www.wikidata.org/entity/Q123185887,2024,DATE,Anora is a 2024 American romantic comedy-drama...,11,15
51,Q133895441,Any Day Now (2024 film),http://www.wikidata.org/entity/Q133895441,2024,DATE,Any Day Now is a 2024 American crime comedy fi...,17,21
53,Q133895441,Any Day Now (2024 film),http://www.wikidata.org/entity/Q133895441,1990,DATE,It is based on the 1990 Isabella Stewart Gardn...,94,98
56,Q124853370,Any Other Way: The Jackie Shane Story,http://www.wikidata.org/entity/Q124853370,2024,DATE,Any Other Way: The Jackie Shane Story is a 202...,43,47
62,Q124853370,Any Other Way: The Jackie Shane Story,http://www.wikidata.org/entity/Q124853370,1971,DATE,"The film is a portrait of Jackie Shane, the pi...",322,326
68,Q112865585,Apartment 7A,http://www.wikidata.org/entity/Q112865585,1968,DATE,Apartment 7A is a 2024 American psychological ...,99,103
74,Q130447150,Apocalypse Z: The Beginning of the End,http://www.wikidata.org/entity/Q130447150,2024,DATE,Apocalypse Z: The Beginning of the End is a 20...,44,48


## Notebook 2 summary

This notebook covered the **text cleaning, named entity recognition, and candidate relation extraction** stage of the project. Its goal was to transform the raw movie summaries collected in Notebook 1 into structured intermediate outputs that could later be converted into RDF triples.

### Main outputs
- **Cleaned summaries:** 580
- **Entity mentions extracted:** 5,262
- **Candidate relations extracted:** 655

### What was done

The notebook first cleaned the movie summaries collected from Wikipedia so that they could be processed more reliably by downstream NLP tools. This step removed unusable or empty entries and kept only summaries with meaningful textual content.

Next, **spaCy NER** was applied to the cleaned summaries to extract entity mentions. The extracted entities included common categories such as:
- **PERSON**
- **ORG**
- **GPE**
- **DATE**
- and other context-dependent named entities

After entity extraction, a relation extraction step was applied to identify candidate subject–predicate–object patterns from the summaries. These candidate relations are not yet final knowledge graph facts, but they provide useful input for the RDF construction stage in the next notebook.

### What worked well

The notebook produced a much larger intermediate dataset than in the earlier smaller run.  
With 580 cleaned summaries, the extraction stage generated a substantial number of reusable entity mentions and candidate relations.

The NER stage successfully captured many useful movie-domain entities such as:
- people involved in films,
- countries and places,
- organizations and companies,
- and temporal mentions.

The relation extraction step also produced a meaningful set of candidate triples that can be reused later when constructing the initial RDF graph.

### Observed limitations

The extracted entities and relations still contain noise, which is expected at this stage:

- Some movie titles or title fragments were incorrectly tagged as entities.
- Some creative works or unusual names were misclassified as **PERSON** or other entity types.
- Many extracted relations came from fallback heuristics, which improves recall but reduces precision.
- Date-like mentions such as years or vague temporal expressions appear frequently but are not always useful for the final graph.
- Some extracted entities are semantically weak or too generic and may need filtering or normalization later.

### Interpretation

These results are good enough for the next stage of the project because Notebook 2 is intended to produce **candidate structured knowledge**, not a perfectly clean final graph.

At this point, the important outcome is that the notebook created:
- a cleaned textual corpus,
- a reusable list of extracted entities,
- and a set of candidate relations for RDF construction.

The remaining ambiguity and noise can be handled later through:
- ontology design,
- graph cleaning,
- alignment,
- and reasoning.

### Main artifacts

This notebook produced the following key files:
- `extracted_entities.csv`
- `extracted_relations.csv`
- `extracted_knowledge.csv`

### Next step

The next notebook converts the outputs of Notebook 1 and Notebook 2 into an **initial RDF knowledge graph**. It combines:
- structured movie metadata,
- extracted entities,
- and candidate relations

to create the first graph that will later be aligned, expanded, reasoned over, and used for KGE and RAG.